# ⚽ FutbolIA — Pipeline de Análisis Táctico con ReID Robusto

**Tracking persistente multimodal:** Kalman CA en metros reales + embeddings MobileNetV3 + histogramas HSV + Algoritmo Húngaro

---
### ▶️ Instrucciones
1. Edita la **Celda 0** con las rutas del video y parámetros que desees. Por defecto viene configurado para correr con los archivos dentro del entorno local `/content/FutbolIA`.
2. Ejecuta las celdas de preparación **Celda 1** y **Celda 2**.
3. Ejecuta **solo la celda del modo que quieras procesar**:
   - **Celda 3a** 🔴: Video ReID + Mapa 2D (con opción de mapa de calor dinámico del jugador en tiempo real).
   - **Celda 3b** 🔵: Videos enfocados por Equipos (bloque táctico, distancias y malla de juego).
   - **Celda 3c** ⚫: Modo Completo (los 4 videos + cuadrícula 2x2).
   - **Celda 3d** 🔍: Escaneo rápido para listar todos los IDs detectados.
   - **Celda 3e** 📊: Generar mapa de calor estático de un jugador en PNG.
4. Visualiza y descarga tus resultados usando las **Celdas 4 y 5**.

---
## 🔧 CELDA 0 — CONFIGURACIÓN DE RUTAS Y PARÁMETROS
---

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║            ⚙️  CONFIGURACIÓN PRINCIPAL — EDITA AQUÍ                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── 📁 CARPETA DE RESPALDO EN DRIVE (Opcional, para guardar resultados) ──
DRIVE_RESPALDO = "/content/drive/MyDrive/Futbol2026"

# ── 🤖 MODELOS YOLO (Rutas locales en Colab) ─────────────────────────────
MODELO_DETECCION   = "/content/FutbolIA/model/best.pt"
MODELO_POSE_CANCHA = "/content/FutbolIA/model/best_pose.pt"

# ── 🎬 VIDEO A PROCESAR (Elige cualquiera de los disponibles en el entorno) ──
# Opciones disponibles en tu entorno:
#   - "/content/FutbolIA/clips/original/clip_01.mp4"  (Recomendado por defecto)
#   - "/content/FutbolIA/clip_02.mp4"
#   - "/content/FutbolIA/clip_03.mp4"
#   - "/content/FutbolIA/12.mp4"
VIDEO_ENTRADA      = "/content/FutbolIA/clips/original/clip_01.mp4"

# ── ⚙️  PARÁMETROS DEL PIPELINE ──────────────────────────────────────────
CONFIANZA_JUGADORES = 0.25   # Umbral detección jugadores (0.0 - 1.0)
TAMAÑO_IMAGEN       = 640    # 640 = rápido | 1280 = más preciso pero más lento

# ── 🔥 MAPA DE CALOR DINÁMICO EN EL VIDEO 2D ─────────────────────────────
JUGADOR_HEATMAP_ID  = None   # Escribe un ID (ej: 12) para ver su mapa de calor dinámico
                             # acumulándose en tiempo real en la cancha 2D del video.
                             # Deja None para no renderizar mapa de calor dinámico.

# ╔══════════════════════════════════════════════════════════════════════╗
# ║         ✅  FIN DE LA CONFIGURACIÓN — No toques lo de abajo         ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os
STEM_VIDEO = os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]

print("╔══════════════════════════════════════════════════════════════╗")
print("║                  CONFIGURACIÓN DE EJECUCIÓN                  ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"  🤖 Detección     : {MODELO_DETECCION}")
print(f"  🏟️  Pose Cancha   : {MODELO_POSE_CANCHA}")
print(f"  🎬 Video entrada  : {VIDEO_ENTRADA}")
print(f"  🏷️  Nombre base   : {STEM_VIDEO}")
print(f"  🎯 Confianza      : {CONFIANZA_JUGADORES}")
print(f"  🔍 Tamaño imagen  : {TAMAÑO_IMAGEN}")
print(f"  🔥 Heatmap video  : ID {JUGADOR_HEATMAP_ID}")
print()
print("  ➡️  Ejecuta las celdas 1 y 2 para preparar el entorno.")

---
## Celda 1 — Instalar dependencias

In [ ]:
!pip install -q ultralytics imageio-ffmpeg tqdm opencv-python-headless numpy scipy torch torchvision pillow
print("✅ Dependencias de Visión por Computadora instaladas.")

---
## Celda 2 — Configurar Entorno de Ejecución (Google Drive & Directorios)

In [ ]:
import os
import shutil
from google.colab import drive

# 1. Intentar montar Google Drive (Opcional, no detiene la ejecución si falla)
DRIVE_MOUNTED = False
try:
    print("📂 Intentando montar Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    DRIVE_MOUNTED = True
    print("✅ Google Drive montado exitosamente.")
except Exception as e:
    print("ℹ️  Google Drive no montado o rechazado (se usarán solo los archivos locales de /content/FutbolIA).")

# 2. Validar estructura del directorio local /content/FutbolIA
cwd = os.getcwd()
print(f"\n📁 Directorio actual de trabajo: {cwd}")

if os.path.exists("/content/FutbolIA/src"):
    print("✅ Estructura de código '/content/FutbolIA/src' detectada correctamente.")
else:
    # En caso de que no esté en /content/FutbolIA, lo clonamos de GitHub
    print("⚠️  No se encontró la carpeta 'src' localmente. Clonando repositorio...")
    !git clone https://github.com/HectrorrVas/FutbolIA.git /content/FutbolIA

# 3. Asegurar directorios de salida
for folder in ["/content/FutbolIA/clips/original",
               "/content/FutbolIA/clips/processed",
               "/content/FutbolIA/output/heatmaps"]:
    os.makedirs(folder, exist_ok=True)
    print(f"  → Directorio asegurado: {folder}")

# 4. Validar existencia de modelos y video seleccionado
archivos_ok = True
for path_verificar, label in [
    (MODELO_DETECCION, "Modelo Detección (best.pt)"),
    (MODELO_POSE_CANCHA, "Modelo Pose Cancha (best_pose.pt)"),
    (VIDEO_ENTRADA, "Video de Entrada")
]:
    if os.path.exists(path_verificar):
        sz = os.path.getsize(path_verificar) / (1024*1024)
        print(f"  ✅ Encontrado: {label} ({sz:.1f} MB)")
    else:
        print(f"  ❌ NO ENCONTRADO: {label} en '{path_verificar}'")
        archivos_ok = False

print()
if archivos_ok:
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  ✅ ¡TODO LISTO! Elige una celda de procesamiento abajo.      ║")
    print("╚══════════════════════════════════════════════════════════════╝")
else:
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  ❌ FALTAN ARCHIVOS. Revisa la Celda 0 o sube tus archivos  ║")
    print("╚══════════════════════════════════════════════════════════════╝")

---
## 🔴 CELDA 3a — PROCESAR: Solo ReID + Mapa 2D

Genera **1 video**: `/content/FutbolIA/clips/processed/{stem}_main.mp4`
- El video original con los **IDs persistentes** de cada jugador dibujados encima.
- El **mapa 2D métrico** de la cancha con las posiciones proyectadas por homografía.
- Si configuraste `JUGADOR_HEATMAP_ID`, el mapa de calor de ese jugador se dibujará en tiempo real sobre la cancha 2D.

> ✅ Esta es la celda más rápida. Ideal para validar el ReID antes de generar todo.

In [ ]:
import sys
sys.path.insert(0, '/content/FutbolIA')

from src.processor import VideoProcessor
import os

nombre_video = os.path.basename(VIDEO_ENTRADA)
stem         = os.path.splitext(nombre_video)[0]
salida       = f"/content/FutbolIA/clips/processed/{stem}_main.mp4"

processor = VideoProcessor(
    model_path=MODELO_DETECCION,
    pose_model_path=MODELO_POSE_CANCHA,
    confidence=CONFIANZA_JUGADORES,
    imgsz=TAMAÑO_IMAGEN
)

processor.process(
    input_video_path=VIDEO_ENTRADA,
    output_video_path=salida,
    mode="reid",
    player_id=JUGADOR_HEATMAP_ID
)

---
## 🔵 CELDA 3b — PROCESAR: Solo Análisis por Equipos

Genera **2 videos**: `{stem}_equipoA.mp4` y `{stem}_equipoB.mp4`
- Vista enfocada en cada equipo con mallas de coordinación, distancias inter-jugador y bloque defensivo (Convex Hull).

In [ ]:
import sys
sys.path.insert(0, '/content/FutbolIA')

from src.processor import VideoProcessor
import os

nombre_video = os.path.basename(VIDEO_ENTRADA)
stem         = os.path.splitext(nombre_video)[0]
salida       = f"/content/FutbolIA/clips/processed/{stem}_equipoA.mp4"

processor = VideoProcessor(
    model_path=MODELO_DETECCION,
    pose_model_path=MODELO_POSE_CANCHA,
    confidence=CONFIANZA_JUGADORES,
    imgsz=TAMAÑO_IMAGEN
)

processor.process(
    input_video_path=VIDEO_ENTRADA,
    output_video_path=salida,
    mode="teams"
)

---
## ⚫ CELDA 3c — PROCESAR: Completo (los 4 videos)

Genera **4 videos**: `_main.mp4`, `_equipoA.mp4`, `_equipoB.mp4` y `_grid.mp4`

> ⚠️ Este modo es el más lento. Úsalo cuando el ReID ya esté validado.

In [ ]:
import sys
sys.path.insert(0, '/content/FutbolIA')

from src.processor import VideoProcessor
import os

nombre_video = os.path.basename(VIDEO_ENTRADA)
stem         = os.path.splitext(nombre_video)[0]
salida       = f"/content/FutbolIA/clips/processed/{stem}_main.mp4"

processor = VideoProcessor(
    model_path=MODELO_DETECCION,
    pose_model_path=MODELO_POSE_CANCHA,
    confidence=CONFIANZA_JUGADORES,
    imgsz=TAMAÑO_IMAGEN
)

processor.process(
    input_video_path=VIDEO_ENTRADA,
    output_video_path=salida,
    mode="full",
    player_id=JUGADOR_HEATMAP_ID
)

---
## 🔍 CELDA 3d — Escanear los IDs de Jugadores del Video

Esta celda realiza un **pase rápido sin renderizar** para listar qué IDs fueron detectados en el video y cuántas veces aparecieron. Úsala para elegir qué ID de jugador procesar después.

In [ ]:
import sys
sys.path.insert(0, '/content/FutbolIA')

from src.processor import VideoProcessor

processor = VideoProcessor(
    model_path=MODELO_DETECCION,
    pose_model_path=MODELO_POSE_CANCHA,
    confidence=CONFIANZA_JUGADORES,
    imgsz=TAMAÑO_IMAGEN
)

# Escanear todo el video rápido
id_registry = processor.scan_player_ids(VIDEO_ENTRADA)

print("\n╔══════════════════════════════════════════════════════════════╗")
print("║                 JUGADORES Y IDs DETECTADOS                   ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"  {'ID':^8} | {'Rol/Equipo':^15} | {'Frames Visibles':^22}")
print("  " + "─" * 50)
roles = {0: "Árbitro", 2: "Equipo A", 3: "Equipo B", 5: "Portero"}
for tid, info in id_registry.items():
    rol = roles.get(info["class_id"], f"Clase {info['class_id']}")
    print(f"  {tid:^8} | {rol:^15} | {info['frames']:^22}")

---
## 📊 CELDA 3e — Mapa de Calor por Jugador Específico (PNG Estático)

Elige un **ID de jugador** del escaneo anterior. La celda procesará el video y generará el mapa de calor en un archivo PNG de alta resolución de forma instantánea en la pantalla.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  👉 ESCRIBE AQUÍ EL ID DEL JUGADOR A PROCESAR                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
JUGADOR_ID = 12

import sys
sys.path.insert(0, '/content/FutbolIA')

from src.processor import VideoProcessor
from IPython.display import Image, display
import os

stem = os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]
out_dir = "/content/FutbolIA/clips/processed"

processor = VideoProcessor(
    model_path=MODELO_DETECCION,
    pose_model_path=MODELO_POSE_CANCHA,
    confidence=CONFIANZA_JUGADORES,
    imgsz=TAMAÑO_IMAGEN
)

processor.process(
    input_video_path=VIDEO_ENTRADA,
    output_video_path=None,
    mode="player_heatmap",
    player_id=JUGADOR_ID
)

path_png = f"{out_dir}/{stem}_heatmap_player_{JUGADOR_ID}.png"
if os.path.exists(path_png):
    print("\n📊 MAPA DE CALOR GENERADO:")
    display(Image(filename=path_png, width=450))
else:
    print(f"❌ ERROR: No se encontró el mapa de calor generado en {path_png}")

---
## Celda 4 — Visualizar los videos procesados

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

def show_video(path, title=""):
    if not os.path.exists(path):
        print(f"⚠️  No encontrado (¿ya procesaste ese modo?): {path}")
        return
    sz = os.path.getsize(path) / (1024*1024)
    data = open(path, 'rb').read()
    url  = "data:video/mp4;base64," + b64encode(data).decode()
    display(HTML(f"""
        <h3 style='color:#ddd;font-family:sans-serif'>{title}
        <small style='color:#888'>({sz:.1f} MB)</small></h3>
        <video width='900' controls style='border-radius:8px;margin-bottom:20px'>
            <source src='{url}' type='video/mp4'>
        </video>
    """))

stem = os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]
base = "/content/FutbolIA/clips/processed"

show_video(f"{base}/{stem}_main.mp4",    "📊 ReID + Mapa 2D Métrico")
show_video(f"{base}/{stem}_equipoA.mp4", "🔵 Análisis Táctico — Equipo A")
show_video(f"{base}/{stem}_equipoB.mp4", "🔴 Análisis Táctico — Equipo B")
show_video(f"{base}/{stem}_grid.mp4",    "⚽ Grid 2x2 — Todas las vistas")

---
## Celda 5 — Descargar los videos y respaldar en Drive

In [ ]:
import os
from google.colab import files
import shutil

stem = os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]
base = "/content/FutbolIA/clips/processed"

videos = [
    (f"{base}/{stem}_main.mp4",    "ReID + Mapa 2D"),
    (f"{base}/{stem}_equipoA.mp4", "Equipo A"),
    (f"{base}/{stem}_equipoB.mp4", "Equipo B"),
    (f"{base}/{stem}_grid.mp4",    "Grid 2x2"),
]

# 1. Descarga local
for path, label in videos:
    if os.path.exists(path):
        print(f"⬇️  Descargando a tu PC: {label}...")
        files.download(path)

# 2. Respaldo automático en Drive (si está montado)
if 'DRIVE_MOUNTED' in globals() and DRIVE_MOUNTED:
    dest_drive = f"{DRIVE_RESPALDO}/clips/processed"
    os.makedirs(dest_drive, exist_ok=True)
    print(f"\n📂 Respaldando en Google Drive en: '{dest_drive}'...")
    for path, label in videos:
        if os.path.exists(path):
            shutil.copy2(path, os.path.join(dest_drive, os.path.basename(path)))
            print(f"  ✅ Respaldado: {label}")